# Embedding keyframe bằng jina-clip-v2 trên T4 x2

Sinh **vector ảnh 1024 chiều** cho keyframe đã tách sẵn của một batch, để nhánh
`dense_visual` khớp truy vấn thẳng với pixel.

**Input:** `Keyframes_<BATCH>/keyframes/<video_id>/*.jpg` trong Kaggle Dataset.
**Output:** `vectors/<video_id>.npy` + `vectors/<video_id>.index.jsonl`,
`manifests/embedding_manifest.jsonl`, `model_info.json`, `_SUCCESS.json`, zip.

## Vì sao jina-clip-v2 chứ không phải CLIP ViT-L/14

Text tower của `openai/clip-vit-large-patch14` là **tiếng Anh thuần**, mà mọi
truy vấn của cuộc thi là tiếng Việt. `jina-clip-v2` dùng text tower
jina-XLM-RoBERTa đa ngữ (chính là `jina-embeddings-v3`) ghép với image tower
EVA02-L/14 ở 512x512. Đây là lý do đáng thử duy nhất — không phải vì nó "mới
hơn".

## Bốn quyết định thiết kế cần biết trước khi sửa

**1. Vector gom thành MỘT `.npy` cho mỗi video, không phải một file JSON cho
mỗi frame.** Bản local ghi `processed/embeddings/<video>/frame_%06d.json`, tiện
khi có 855 keyframe. Full data là hàng trăm nghìn frame — bằng đó file nhỏ sẽ
làm chậm cả việc zip lẫn việc giải nén, và vượt giới hạn output của Kaggle.
Cell cuối có `EXPLODE_PER_FRAME` để bung ra layout cũ khi cần cắm thẳng vào
`build_frame_vector_rows` của bản local.

**2. Lưu float16.** Vector đã L2-normalize nên dải giá trị nằm gọn trong
[-1, 1]; fp16 đủ chính xác cho cosine mà giảm nửa dung lượng. Đọc ra thì cast
lại float32 trước khi nhân.

**3. Chuẩn hoá L2 LẠI ở phía notebook** dù `encode_image` nói đã chuẩn hoá.
Cả index lẫn text tower đều coi dot product = cosine; lệch một bên là điểm số
vô nghĩa mà không có gì báo lỗi.

**4. Mỗi GPU một process, ghim bằng `CUDA_VISIBLE_DEVICES`.** Giống notebook
caption/OCR. Shard theo chỉ số frame nên hai worker không bao giờ đụng nhau.

## Thứ tự tác vụ

1. `discover_keyframes` -> 2. `download_weights` -> 3. `smoke_test` ->
4. `embed_all` -> 5. `merge_shards` -> 6. `package_zip` / `validate_output`


In [ ]:
# Kaggle đã có sẵn torch bản CUDA — cài lại torch từ PyPI có thể kéo về bản
# CPU-only và làm hỏng GPU cho cả phiên. Chỉ thêm thứ jina-clip cần:
#   timm    -> image tower EVA02 nằm trong timm
#   einops  -> code remote của jina-xlm-roberta dùng
%pip install -q --no-cache-dir "transformers>=4.51,<5" timm einops


## 1. Cấu hình


In [ ]:
from pathlib import Path
import os

# ===================== ĐỔI DUY NHẤT DÒNG NÀY =====================
BATCH = "L21_a"          # L21_a, L21_b, L22_a, ...
# =================================================================

# Để trống = tự dò `Keyframes_<BATCH>` trong /kaggle/input.
KEYFRAME_ROOT_OVERRIDE = os.environ.get("AIC_KEYFRAME_ROOT", "")

MODEL_ID = "jinaai/jina-clip-v2"
EMBEDDING_NAME = "jina_clip_v2"
EMBED_DIM = 1024              # projection_dim của jina-clip-v2
TRUNCATE_DIM = 0              # >0 = cắt Matryoshka (32/64/128/256/512/768); 0 = giữ 1024

STAGE_NAME = f"embedding_{BATCH}"
OUTPUT_ROOT = Path(f"/kaggle/working/aic_stage_embedding_{BATCH}")
ZIP_PATH = Path(f"/kaggle/working/embedding_{BATCH}_output.zip")

DTYPE = "float16"             # T4 là Turing: fp16 nhanh và ổn định, bf16 không có tensor core
BATCH_SIZE = 16               # ảnh mỗi lần forward, TRÊN MỖI GPU. OOM thì worker tự chia đôi.
MAX_FRAMES = 0                # 0 = tất cả; đặt số nhỏ để thử nhanh
SMOKE_FRAMES = 8              # số frame MỖI GPU cho lần chạy thử
PACK_VERSION = "jina-clip-v2-1.0.0"
EXPLODE_PER_FRAME = False     # True = ghi thêm layout JSON-mỗi-frame của bản local

SHARD_DIR = OUTPUT_ROOT / "shards"
TASKS_PATH = OUTPUT_ROOT / "tasks.jsonl"
VECTOR_DIR = OUTPUT_ROOT / "vectors"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in (SHARD_DIR, VECTOR_DIR, MANIFEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("BATCH          =", BATCH)
print("MODEL_ID       =", MODEL_ID)
print("OUTPUT_ROOT    =", OUTPUT_ROOT)
print("EMBEDDING_NAME =", EMBEDDING_NAME)


## 2. Tìm thư mục keyframe & kiểm tra trước khi tốn GPU

Dò và ĐẾM trước, vì phát hiện sai đường dẫn sau 40 phút GPU là mất trắng cả
phiên. `frame_idx` lấy từ TÊN FILE (`frame_%06d.jpg`) chứ không phải thứ tự
liệt kê — thứ tự `glob` không đảm bảo, và `frame_idx` là khoá nối với
`keyframes.jsonl` của export.


In [ ]:
import json
import re

FRAME_RE = re.compile(r"frame_(\d+)\.(?:jpg|jpeg|png)$", re.IGNORECASE)


def find_keyframe_root(batch: str, override: str = "") -> Path:
    if override:
        root = Path(override)
        if not root.exists():
            raise FileNotFoundError(f"AIC_KEYFRAME_ROOT trỏ vào chỗ không tồn tại: {root}")
        return root
    candidates = sorted(Path("/kaggle/input").glob(f"*{batch}*"))
    if not candidates:
        available = sorted(p.name for p in Path("/kaggle/input").iterdir())
        raise FileNotFoundError(
            f"không thấy dataset nào khớp {batch!r} trong /kaggle/input. Có: {available}"
        )
    for candidate in candidates:
        nested = candidate / "keyframes"
        if nested.is_dir():
            return nested
    return candidates[0]


def discover_keyframes(root: Path, max_frames: int = 0) -> list[dict]:
    """Trả về task đã sắp xếp ổn định: (video_id, frame_idx).

    Sắp xếp tường minh để hai lần chạy chia shard giống hệt nhau — chạy lại sau
    khi gián đoạn mới ghép đúng phần đã có.
    """
    tasks: list[dict] = []
    for image_path in root.rglob("*"):
        match = FRAME_RE.search(image_path.name)
        if match is None:
            continue
        tasks.append({
            "video_id": image_path.parent.name,
            "frame_idx": int(match.group(1)),
            "image_path": str(image_path),
        })
    tasks.sort(key=lambda item: (item["video_id"], item["frame_idx"]))
    if max_frames:
        tasks = tasks[:max_frames]
    return tasks


KEYFRAME_ROOT = find_keyframe_root(BATCH, KEYFRAME_ROOT_OVERRIDE)
tasks = discover_keyframes(KEYFRAME_ROOT, MAX_FRAMES)
if not tasks:
    raise SystemExit(f"không tìm thấy keyframe nào dưới {KEYFRAME_ROOT}")

TASKS_PATH.write_text(
    "".join(json.dumps(task, ensure_ascii=False) + "\n" for task in tasks), encoding="utf-8"
)

from collections import Counter

per_video = Counter(task["video_id"] for task in tasks)
print("KEYFRAME_ROOT =", KEYFRAME_ROOT)
print(f"{len(tasks)} keyframe / {len(per_video)} video")
for video_id, count in sorted(per_video.items())[:10]:
    print(f"  {video_id:14s} {count:6d}")
if len(per_video) > 10:
    print(f"  … còn {len(per_video) - 10} video")


## 3. Đếm GPU & tải trọng số một lần

Tải TRƯỚC khi fork worker: hai process cùng gọi `snapshot_download` vào cùng
cache sẽ đua nhau trên cùng file và thỉnh thoảng để lại blob dở dang.


In [ ]:
import torch
from huggingface_hub import snapshot_download

NUM_SHARDS = max(1, torch.cuda.device_count())
print("GPU thấy được:", NUM_SHARDS)
for index in range(NUM_SHARDS):
    print(" ", index, torch.cuda.get_device_name(index))

MODEL_DIR = Path(snapshot_download(
    MODEL_ID,
    ignore_patterns=["onnx/*", "*.onnx", "*.onnx_data", "openvino/*"],
))
print("MODEL_DIR =", MODEL_DIR)

# jina-clip nạp code từ MỘT REPO RỜI (`jinaai/jina-clip-implementation`) qua
# auto_map. Kéo sẵn ở đây, nếu không worker chạy với HF_HUB_OFFLINE=1 sẽ chết
# ngay lúc from_pretrained mà thông báo lỗi chẳng liên quan gì tới mạng.
IMPL_DIR = Path(snapshot_download("jinaai/jina-clip-implementation", allow_patterns=["*.py", "*.json"]))
print("IMPL_DIR  =", IMPL_DIR)


## 4. Worker

Một process cho mỗi GPU. Ghi shard dạng `.npy` + `.jsonl` chỉ mục, flush theo
lô nên gián đoạn giữa chừng vẫn giữ được phần đã xong.


In [ ]:
%%writefile /kaggle/working/aic_embed_worker.py
"""Embed keyframe bằng jina-clip-v2 cho MỘT shard / MỘT GPU."""
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from transformers import AutoModel


def load_tasks(path: Path, shard: int, num_shards: int, limit: int) -> list[dict]:
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    mine = [row for index, row in enumerate(rows) if index % num_shards == shard]
    return mine[:limit] if limit else mine


def encode_batch(model, images: list[Image.Image], truncate_dim: int) -> np.ndarray:
    """Một lô ảnh -> ma trận đã L2-normalize.

    OOM thì chia đôi và thử lại, đệ quy tới lô 1 ảnh. T4 16GB ở 512x512 chịu
    được lô lớn, nhưng ảnh nguồn kích thước rất lệch nhau nên một lô xui vẫn
    có thể vượt — chia đôi rẻ hơn nhiều so với hạ BATCH_SIZE cho cả lượt chạy.
    """
    try:
        with torch.no_grad():
            kwargs = {"truncate_dim": truncate_dim} if truncate_dim else {}
            raw = model.encode_image(images, batch_size=len(images), **kwargs)
    except torch.cuda.OutOfMemoryError:
        if len(images) == 1:
            raise
        torch.cuda.empty_cache()
        middle = len(images) // 2
        return np.vstack([
            encode_batch(model, images[:middle], truncate_dim),
            encode_batch(model, images[middle:], truncate_dim),
        ])
    matrix = np.asarray(raw, dtype="float32")
    norms = np.linalg.norm(matrix, axis=-1, keepdims=True)
    return matrix / np.clip(norms, 1e-9, None)


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--tasks", type=Path, required=True)
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--progress", type=Path, required=True)
    parser.add_argument("--shard", type=int, required=True)
    parser.add_argument("--num-shards", type=int, required=True)
    parser.add_argument("--model", type=str, required=True)
    parser.add_argument("--dtype", default="float16")
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--truncate-dim", type=int, default=0)
    parser.add_argument("--limit", type=int, default=0)
    args = parser.parse_args()

    tasks = load_tasks(args.tasks, args.shard, args.num_shards, args.limit)
    dtype = {"float16": torch.float16, "float32": torch.float32}[args.dtype]
    model = AutoModel.from_pretrained(args.model, trust_remote_code=True, torch_dtype=dtype)
    model = model.to("cuda").eval()

    vectors: list[np.ndarray] = []
    index_rows: list[dict] = []
    started = time.monotonic()
    failed = 0

    def write_progress(done: int) -> None:
        elapsed = time.monotonic() - started
        args.progress.write_text(json.dumps({
            "done": done, "found": len(index_rows), "failed": failed,
            "total": len(tasks),
            "rate_sec_per_frame": (elapsed / done) if done else None,
        }), encoding="utf-8")

    def flush() -> None:
        if vectors:
            np.save(args.out.with_suffix(".npy"), np.vstack(vectors).astype("float16"))
        args.out.with_suffix(".index.jsonl").write_text(
            "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in index_rows),
            encoding="utf-8",
        )

    for start in range(0, len(tasks), args.batch_size):
        chunk = tasks[start : start + args.batch_size]
        images, kept = [], []
        for task in chunk:
            try:
                images.append(Image.open(task["image_path"]).convert("RGB"))
                kept.append(task)
            except Exception:  # ảnh hỏng: bỏ qua, đừng giết cả shard
                failed += 1
        if images:
            matrix = encode_batch(model, images, args.truncate_dim)
            vectors.append(matrix)
            for task in kept:
                index_rows.append({"video_id": task["video_id"], "frame_idx": task["frame_idx"]})
        write_progress(min(start + args.batch_size, len(tasks)))
        if (start // args.batch_size) % 20 == 0:
            flush()

    flush()
    write_progress(len(tasks))


if __name__ == "__main__":
    main()


## 5. Chạy thử vài frame trước khi tốn cả phiên


In [ ]:
import subprocess
import sys
import time
from typing import Any


WORKER_PATH = "/kaggle/working/aic_embed_worker.py"


def launch_workers(*, tag: str, limit: int = 0) -> list[dict[str, Any]]:
    """Mỗi GPU một process. `tag` tách hẳn file của lần chạy thử và lần chạy thật."""
    processes = []
    for shard in range(NUM_SHARDS):
        out_path = SHARD_DIR / f"{tag}_shard{shard}"
        command = [
            sys.executable, WORKER_PATH,
            "--tasks", str(TASKS_PATH),
            "--out", str(out_path),
            "--progress", str(SHARD_DIR / f"{tag}_progress{shard}.json"),
            "--shard", str(shard),
            "--num-shards", str(NUM_SHARDS),
            "--model", str(MODEL_DIR),
            "--dtype", DTYPE,
            "--batch-size", str(BATCH_SIZE),
            "--truncate-dim", str(TRUNCATE_DIM),
            "--limit", str(limit),
        ]
        environment = dict(
            os.environ,
            CUDA_VISIBLE_DEVICES=str(shard),   # ghim process này vào đúng 1 GPU
            TOKENIZERS_PARALLELISM="false",
            HF_HUB_OFFLINE="1",                # trọng số + code remote đã tải ở cell 3
        )
        log_path = SHARD_DIR / f"{tag}_worker{shard}.log"
        handle = log_path.open("w", encoding="utf-8")
        processes.append({
            "shard": shard,
            "proc": subprocess.Popen(command, stdout=handle, stderr=subprocess.STDOUT, env=environment),
            "log_handle": handle,
            "log_path": log_path,
            "progress": SHARD_DIR / f"{tag}_progress{shard}.json",
            "out": out_path,
        })
    return processes


def read_json_safe(path: Path) -> dict:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}


def monitor(processes: list[dict[str, Any]], total: int, label: str) -> dict[str, int]:
    totals = {"done": 0, "found": 0, "failed": 0}
    while True:
        states = [read_json_safe(entry["progress"]) for entry in processes]
        totals = {key: sum(state.get(key, 0) for state in states)
                  for key in ("done", "found", "failed")}
        rates = [s["rate_sec_per_frame"] for s in states if s.get("rate_sec_per_frame")]
        eta = (total - totals["done"]) * (sum(rates) / len(rates)) / len(processes) if rates else 0
        print(f"\r{label}: {totals['done']}/{total} ok={totals['found']} "
              f"lỗi={totals['failed']} ETA={eta/60:.1f}m", end="", flush=True)
        if all(entry["proc"].poll() is not None for entry in processes):
            break
        time.sleep(3)
    print()
    for entry in processes:
        entry["log_handle"].close()
        # Worker chết vì lỗi khác OOM sẽ để tiến độ đứng im chứ không báo gì ra
        # notebook — đọc lại đuôi log ở đây để lỗi thật nổi lên thay vì bị nuốt.
        if entry["proc"].returncode != 0:
            tail = entry["log_path"].read_text(encoding="utf-8")[-2000:]
            raise RuntimeError(f"worker shard={entry['shard']} thoát {entry['proc'].returncode}:\n{tail}")
    return totals


for stale in SHARD_DIR.glob("smoke_*"):
    stale.unlink()
smoke = launch_workers(tag="smoke", limit=SMOKE_FRAMES)
print(monitor(smoke, SMOKE_FRAMES * NUM_SHARDS, "smoke"))


In [ ]:
import numpy as np

# Kiểm tra ngay trên kết quả smoke, TRƯỚC khi chạy toàn bộ. Ba thứ này bắt được
# gần hết lỗi im lặng của một pipeline embedding.
probe = np.load(SHARD_DIR / "smoke_shard0.npy").astype("float32")
norms = np.linalg.norm(probe, axis=1)
pairs = probe @ probe.T
off_diagonal = pairs[~np.eye(len(pairs), dtype=bool)]
print("shape        :", probe.shape, "(phải là (n,", EMBED_DIM if not TRUNCATE_DIM else TRUNCATE_DIM, "))")
print("NaN          :", int(np.isnan(probe).sum()), "(phải là 0)")
print("norm         :", f"[{norms.min():.4f}, {norms.max():.4f}]", "(phải là ~1.0)")
print("cosine cặp   :", f"mean={off_diagonal.mean():.3f} max={off_diagonal.max():.3f}")
print("  -> mean gần 1.0 nghĩa là mọi ảnh ra cùng một vector: model nạp hỏng")
print("  -> shape sai chiều nghĩa là TRUNCATE_DIM/model khác điều đang tưởng")


## 6. Chạy toàn bộ


In [ ]:
started_all = time.monotonic()
workers = launch_workers(tag="full")
totals = monitor(workers, len(tasks), "embed_all")
print(totals, f"trong {(time.monotonic() - started_all)/60:.1f} phút")


## 7. Gộp shard

Shard chia theo `index % num_shards` nên thứ tự toàn cục bị trộn. Gộp lại theo
`(video_id, frame_idx)` để mỗi video thành MỘT ma trận có thứ tự xác định.


In [ ]:
def merge_shards() -> dict[str, int]:
    by_video: dict[str, list[tuple[int, np.ndarray]]] = {}
    for shard in range(NUM_SHARDS):
        matrix_path = SHARD_DIR / f"full_shard{shard}.npy"
        index_path = SHARD_DIR / f"full_shard{shard}.index.jsonl"
        if not matrix_path.exists():
            continue
        matrix = np.load(matrix_path)
        rows = [json.loads(line) for line in index_path.read_text(encoding="utf-8").splitlines() if line.strip()]
        if len(rows) != len(matrix):
            raise ValueError(
                f"shard {shard} lệch: {len(rows)} dòng chỉ mục nhưng {len(matrix)} vector"
            )
        for row, vector in zip(rows, matrix):
            by_video.setdefault(row["video_id"], []).append((row["frame_idx"], vector))

    counts: dict[str, int] = {}
    manifest_rows: list[dict] = []
    for video_id, items in sorted(by_video.items()):
        items.sort(key=lambda pair: pair[0])
        matrix = np.vstack([vector for _idx, vector in items]).astype("float16")
        np.save(VECTOR_DIR / f"{video_id}.npy", matrix)
        (VECTOR_DIR / f"{video_id}.index.jsonl").write_text(
            "".join(json.dumps({"frame_idx": int(idx), "row": position}) + "\n"
                    for position, (idx, _v) in enumerate(items)),
            encoding="utf-8",
        )
        counts[video_id] = len(items)
        for position, (frame_idx, _vector) in enumerate(items):
            manifest_rows.append({
                "video_id": video_id,
                "frame_idx": int(frame_idx),
                "embedding_refs": [{
                    "embedding_name": EMBEDDING_NAME,
                    "modality": "image",
                    "model_name": MODEL_ID,
                    "model_revision": None,
                    "dimension": int(matrix.shape[1]),
                    "normalized": True,
                    "storage_locations": [{
                        "backend": "file",
                        "vector_id": f"{video_id}_F{int(frame_idx):06d}",
                        "index_name": EMBEDDING_NAME,
                        # Trỏ vào MỘT HÀNG của ma trận video, không phải một file
                        # riêng. Bản local đọc file-mỗi-vector; xem cell 8 nếu
                        # cần layout đó.
                        "vector_uri": f"vectors/{video_id}.npy#{position}",
                    }],
                }],
            })

    (MANIFEST_DIR / "embedding_manifest.jsonl").write_text(
        "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in manifest_rows),
        encoding="utf-8",
    )
    (OUTPUT_ROOT / "model_info.json").write_text(json.dumps({
        "component": "embedding", "model": MODEL_ID, "pack_version": PACK_VERSION,
        "embedding_name": EMBEDDING_NAME,
        "dimension": TRUNCATE_DIM or EMBED_DIM, "normalized": True, "stored_dtype": "float16",
    }, ensure_ascii=False), encoding="utf-8")
    (OUTPUT_ROOT / "_SUCCESS.json").write_text(json.dumps({
        "status": "success", "stage": "embedding", "count": len(manifest_rows),
    }, ensure_ascii=False), encoding="utf-8")
    return counts


counts = merge_shards()
total_vectors = sum(counts.values())
print(f"{total_vectors} vector / {len(counts)} video")
if total_vectors != len(tasks):
    print(f"CẢNH BÁO: {len(tasks) - total_vectors} keyframe không có vector (ảnh hỏng?)")


## 8. (Tuỳ chọn) Bung ra layout JSON-mỗi-frame của bản local

Chỉ bật khi muốn cắm thẳng vào `online/adapters/frame_vector_store.py` của bản
chạy local. Với full data thì để `False`: số file sẽ bằng số keyframe.


In [ ]:
if EXPLODE_PER_FRAME:
    exploded = OUTPUT_ROOT / "processed" / "embeddings" / EMBEDDING_NAME
    written = 0
    for video_id in sorted(counts):
        matrix = np.load(VECTOR_DIR / f"{video_id}.npy").astype("float32")
        index_rows = [json.loads(line) for line in
                      (VECTOR_DIR / f"{video_id}.index.jsonl").read_text(encoding="utf-8").splitlines()
                      if line.strip()]
        target = exploded / video_id
        target.mkdir(parents=True, exist_ok=True)
        for row in index_rows:
            (target / f"frame_{row['frame_idx']:06d}.json").write_text(
                json.dumps([float(value) for value in matrix[row["row"]]]), encoding="utf-8"
            )
            written += 1
    print(f"đã bung {written} file vector -> {exploded}")
else:
    print("EXPLODE_PER_FRAME=False — chỉ giữ layout .npy gọn")


## 9. Đóng gói & kiểm tra output


In [ ]:
import shutil

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
shutil.make_archive(str(ZIP_PATH.with_suffix("")), "zip", root_dir=OUTPUT_ROOT)
print(f"{ZIP_PATH}  ({ZIP_PATH.stat().st_size / 1e6:.1f} MB)")

# Kiểm tra lần cuối trên dữ liệu ĐÃ GỘP, không phải trên smoke: bước gộp có thể
# tự làm hỏng thứ tự mà từng shard vẫn đúng.
sample_video = sorted(counts)[0]
sample = np.load(VECTOR_DIR / f"{sample_video}.npy").astype("float32")
norms = np.linalg.norm(sample, axis=1)
rng = np.random.default_rng(0)
left, right = rng.integers(0, len(sample), 2000), rng.integers(0, len(sample), 2000)
pair_cos = (sample[left] * sample[right]).sum(1)
print(f"{sample_video}: shape={sample.shape} NaN={int(np.isnan(sample).sum())} "
      f"norm=[{norms.min():.4f},{norms.max():.4f}]")
print(f"  cosine cặp ngẫu nhiên: mean={pair_cos.mean():.3f} p95={np.percentile(pair_cos,95):.3f}")
print("  -> mean gần 1.0 = vector sụp đổ, index sẽ vô dụng dù mọi thứ khác trông đúng")
